In [ ]:
%matplotlib inline 

**Todo**
- Write out results for one part of sub-rayliegh and supershear
- pick an example and then explain all the phenomena seen in the example
- basically talk about everything that is necessary according to the example

In [ ]:
# Imports
import os
import pathlib
import urllib.request

# Site name for Salvus Flow. Uses env var if available, otherwise falls back to local site name.
SALVUS_FLOW_SITE_NAME = os.environ.get("SALVUS_FLOW_SITE_NAME", "salome_remote_2")
PROJECT_DIR = "simulation_wavefield_custom_stf_rolled" 

# Conservative default to reduce license-seat demand during unstable license-server periods.
RANKS_PER_JOB = 4


def check_license_server_reachable(url="https://l.mondaic.com/licensing_server", timeout=10):
    """Fail fast if licensing endpoint is unreachable to avoid long hanging jobs."""
    try:
        with urllib.request.urlopen(url, timeout=timeout):
            return True
    except Exception as exc:
        raise RuntimeError(
            f"Licensing server not reachable ({url}). Retry later or check network/VPN. Original error: {exc}"
        ) from exc


# Add code to keep .gitignore updated to ignore salvus files
gitignore_path = pathlib.Path("..") / ".gitignore"
with open(gitignore_path, "r+") as f:
    contents = f.read()
    if PROJECT_DIR not in contents:
        f.write(f"\n{PROJECT_DIR}/\n")


import numpy as np
import salvus.namespace as sn
import salvus.flow.simple_config as sc
import xarray as xr
import salvus.namespace as sn
from salvus.project.tools.processing import block_processing
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import matplotlib.pyplot as plt
from matplotlib import animation
import obspy

# For wavefield output code
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers import wavefield_output
import salvus.flow.api

#for plotting of wiggles, traces 
from scipy import signal

# For animation plot
from IPython.display import HTML
from matplotlib.collections import PolyCollection
import threading
import matplotlib
matplotlib.use("Agg")
from scipy.interpolate import griddata

In [ ]:
# Setup of the model domain as a box (same as research module)
domain_2d = sn.domain.dim2.BoxDomain(x0=0, x1=400, y0=0, y1=3)
p = sn.Project.from_domain(path=PROJECT_DIR, domain=domain_2d, load_if_exists=True)

In [ ]:


# Simulation constants
f0 = 20.0
sampling_rate = 10000.0
dt = 1.0 / sampling_rate

step =0.1
x_positions = np.arange(30.0, 270.0, step)
target_vprop = 90.0
delay_between_sources = step / target_vprop
print(f"Time step between sources: {delay_between_sources:.4f} s")

y_src = 2.625

# Shared time setup (half width can be the same for all the wavelets )
half_width = 0.08
pre_delay = half_width

last_source_delay = (len(x_positions) - 1) * delay_between_sources
t_max = pre_delay + last_source_delay + half_width + 0.5
t_sim = np.arange(0, t_max, dt)
print(f"t_sim length: {len(t_sim)} samples, duration: {t_sim[-1]:.3f} s")

t_local = np.arange(-half_width, half_width, dt)
half_samples = len(t_local) // 2

# Define Bases of wavelets
# Ricker 
ricker_base = (
    (1.0 - 2.0 * (np.pi * f0 * t_local) ** 2)
    * np.exp(-((np.pi * f0 * t_local) ** 2))
)

# Morlet
morlet_base = (
    np.exp(-((t_local / half_width) ** 2)) 
    * np.cos(2 * np.pi * f0 * t_local)
)

# Gaussian
gaussian_base = np.exp(-((t_local / half_width) ** 2))


# Weigths setup
base_mxx =  1.0
base_myy = -1.0
base_mxy =  1.0

srcs = []

random_weight_xx = np.ones(len(x_positions)) 
random_weight_yy = np.ones(len(x_positions))
random_weight_xy = np.ones(len(x_positions))

weight_array = np.array([random_weight_xx, random_weight_yy, random_weight_xy]).T

# Main source setup
for i, x_src in enumerate(x_positions):
    center_time = pre_delay + i * delay_between_sources
    center_sample = int(round(center_time * sampling_rate))

    start_idx = center_sample - half_samples
    end_idx   = center_sample + half_samples

    # Initialize empty arrays for the delayed wavelets
    ricker_delayed = np.zeros(len(t_sim))
    morlet_delayed = np.zeros(len(t_sim))
    gaussian_delayed = np.zeros(len(t_sim))

    if end_idx > 0 and start_idx < len(t_sim):
        sim_start = max(0, start_idx)
        sim_end   = min(len(t_sim), end_idx)
        wav_start = max(0, -start_idx)
        wav_end   = wav_start + (sim_end - sim_start)
        
        # Shift all three wavelets into the simulation time window
        ricker_delayed[sim_start:sim_end]   = ricker_base[wav_start:wav_end]
        morlet_delayed[sim_start:sim_end]   = morlet_base[wav_start:wav_end]
        gaussian_delayed[sim_start:sim_end] = gaussian_base[wav_start:wav_end]

    # Sanity check for clipping (we only need to check one since they share dimensions)
    n_nonzero = np.count_nonzero(ricker_delayed)
    expected  = len(ricker_base)
    if n_nonzero < expected - 2:
        print(f"  WARNING source {i}: only {n_nonzero}/{expected} samples written — wavelets clipped!")

    # Construct stf array
    stf_vector_array = np.array([
        ricker_delayed   * (base_mxx * weight_array[i, 0]),
        morlet_delayed   * (base_myy * weight_array[i, 1]),
        gaussian_delayed * (base_mxy * weight_array[i, 2]),
    ])

    # stf setup
    stf = sc.stf.Custom.from_array(
        array=stf_vector_array,
        sampling_rate_in_hertz=sampling_rate,
        start_time_in_seconds=0.0, 
    )

    # plotting_steps = np.arange(0, len(x_positions), 5)
    # if i in plotting_steps:
    #     print(f"Source {i} at x={x_src:.1f} m | delay={center_time:.4f} s | "
    #           f"weight xx:{weight_array[i,0]:.4f} yy:{weight_array[i,1]:.4f} xy:{weight_array[i,2]:.4f}")
        
       
    #     stf.plot()
    #     display(plt.gcf())
    #     plt.close()

    src = sc.source.cartesian.MomentTensorPoint2D(
        x=x_src,
        y=y_src,
        mxx=1.0,
        myy=1.0,
        mxy=1.0,
        source_time_function=stf,
    )
    srcs.append(src)

print(f"\nGenerated {len(srcs)} sources.")
print(f"First source centred at: {pre_delay:.4f} s")
print(f"Last  source centred at: {pre_delay + (len(srcs)-1)*delay_between_sources:.4f} s")
print(f"t_sim spans 0 to {t_sim[-1]:.3f} s")

In [ ]:
srcs

In [ ]:
import h5py

def verify_and_plot_src_components(src_list, target_x=30.0):
    """
    Finds the Mxx, Myy, and Mxy sources matching a target x-coordinate,
    reads their hidden HDF5 arrays, and plots them to verify correct appending.
    """
    # Helper to safely handle both dictionaries and custom Salvus objects
    def get_val(obj, key):
        if isinstance(obj, dict):
            return obj.get(key)
        return getattr(obj, key, None)

    # Dictionaries to store the waveforms we extract
    extracted_waveforms = {"Mxx": None, "Myy": None, "Mxy": None}
    
    print(f"--- Extracting STF components for sources at x = {target_x} m ---")
    
    for s in src_list:
        # Extract location coordinates
        loc = get_val(s, 'location') or [get_val(s, 'x'), get_val(s, 'y')]
        if loc is None or len(loc) < 1:
            continue
            
        # Match only the target x position (using close tolerance for floats)
        if np.isclose(loc[0], target_x):
            # Extract spatial weights/tensor configurations
            weights = get_val(s, 'spatial_weights')
            if weights is None:
                weights = [get_val(s, 'mxx'), get_val(s, 'myy'), get_val(s, 'mxy')]
            
            # Map weights to their mathematical component name
            if np.allclose(weights, [1.0, 0.0, 0.0]):
                comp_name = "Mxx"
                expected_wave = "Ricker"
            elif np.allclose(weights, [0.0, 1.0, 0.0]):
                comp_name = "Myy"
                expected_wave = "Morlet"
            elif np.allclose(weights, [0.0, 0.0, 1.0]):
                comp_name = "Mxy"
                expected_wave = "Gaussian"
            else:
                continue

            # Dig into the source time function block to get the filepath
            stf_info = get_val(s, 'source_time_function')
            filename = get_val(stf_info, 'filename')
            dataset_name = get_val(stf_info, 'dataset_name') or '/stf'
            
            # Open the temporary HDF5 file and extract the raw array
            try:
                with h5py.File(filename, 'r') as h5_file:
                    data = h5_file[dataset_name][:]
                    # Salvus can store traces as 2D arrays (e.g. shape [1, samples]). Squeeze it.
                    extracted_waveforms[comp_name] = np.squeeze(data)
                print(f" -> Successfully read {comp_name} ({expected_wave}) from: {filename.split('/')[-2]}")
            except Exception as e:
                print(f"  Failed to read {comp_name} file: {e}")

    # Plotting the pashe 
    fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    components = ["Mxx", "Myy", "Mxy"]
    labels = ["Mxx (Should be Ricker)", "Myy (Should be Morlet)", "Mxy (Should be Gaussian)"]
    colors = ["crimson", "dodgerblue", "forestgreen"]

    for ax, comp, label, color in zip(axes, components, labels, colors):
        wave = extracted_waveforms[comp]
        if wave is not None:
            # If a 2D multi-component vector slice is still present, grab the first row
            if wave.ndim > 1:
                wave = wave[0]
                
            ax.plot(wave, color=color, linewidth=1.5, label=label)
            ax.set_ylabel("Amplitude")
            ax.legend(loc="upper right")
            ax.grid(True, linestyle="--", alpha=0.5)
        else:
            ax.text(0.5, 0.5, f"Missing component: {comp}", ha='center', va='center', color='red')
            ax.set_ylabel("Amplitude")

    axes[-1].set_xlabel("Time Samples")
    plt.suptitle(f"Verification of Independent STF Waveforms at x={target_x}m", fontsize=13, fontweight='bold')
    plt.tight_layout()
    display(plt.gcf())
    plt.close()

# Run the function on your sources list
verify_and_plot_src_components(srcs, target_x=30.0)

In [ ]:
# fig,ax = plt.subplots()
# array=(1.0 - 2.0 * (np.pi * f0 * t) ** 2) * np.exp(-((np.pi * f0 * t) ** 2)),

# ax.plot(array[0])
# ax.plot(np.roll(array[0], 10))
# display(fig)
# #plt.close(fig)

In [ ]:
# array[0]

In [ ]:

# Layered model setup: three layers ordered as snow, slab, air (top to bottom).

x_min, x_max = 0.0, 400.0

# Geometry (y-coordinates: higher = higher depth, measured downward):
# Snow: from y=3.0 m (top) to y=2.25 m
# Slab: from y=2.25 m to y=1.5 m  
# Air:  from y=1.5 m to y=0.0 m (bottom)

slab_top = 3.0
slab_bottom = 2.25
wl_top = 2.25
wl_bottom = 1.5
air_top = 1.5
air_bottom = 0.0

# Boundaries from top to bottom -> 3 layers.
# Must be strictly ordered: top to bottom (decreasing y).
layers_x = [
    np.array([x_min, x_max]),  # snow top boundary
    np.array([x_min, x_max]),  # snow-slab interface
    np.array([x_min, x_max]),  # slab-air interface
    np.array([x_min, x_max]),  # air bottom boundary
]
layers_y = [
    np.array([slab_top, slab_top]),      # y = 3.0
    np.array([slab_bottom, slab_bottom]),  # y = 2.25
    np.array([wl_bottom, wl_bottom]),  # y = 1.5
    np.array([air_bottom, air_bottom]),    # y = 0.0
]

# Material parameters by region index [snow, slab, air].
vp = np.array([332.0, 300.0, 300.0]) 
vs = np.array([150.0, 150.0, 0.0])   
rho = np.array([180.0, 180.0, 1.225])

interpolation_styles = ["linear"] * len(layers_x)
splines = sn.toolbox.get_interpolating_splines(layers_x, layers_y, kind=interpolation_styles)
 
max_frequency = np.percentile([vp[0], vp[1], vp[2]], 95) # set this as the 95th percentile of the expected frequency content
print(f"Max frequency for meshing: {max_frequency:.1f} Hz")
# One per layer pair; last value keeps meshing stable above acoustic air.
slowest_velocities = np.array([150.0, 150.0,  150.0])

mesh, bnd = sn.toolbox.generate_mesh_from_splines_2d(
    x_min=x_min,
    x_max=x_max,
    splines=splines,
    elements_per_wavelength=4,
    maximum_frequency=max_frequency,
    use_refinements=True,
    slowest_velocities=slowest_velocities,
    absorbing_boundaries=(["x0", "x1", "y0"], 10.0),
)
mesh = np.sum(mesh)
mesh.attach_global_variable("max_dist_ABC", bnd)
mesh.attach_global_variable("ABC_side_sets", ", ".join(["x0", "x1", "y0"]))
mesh.attach_global_variable("ABC_vel", float(min(vs[vs > 0])))
mesh.attach_global_variable("ABC_freq", max_frequency / 2.0)
mesh.attach_global_variable("ABC_nwave", 5.0)

nodes = mesh.get_element_nodes()[:, :, 0]
vp_a, vs_a, ro_a = np.ones((3, *nodes.shape))
for _i, (vp_val, vs_val, ro_val) in enumerate(zip(vp, vs, rho)):
    idx = np.where(mesh.elemental_fields["region"] == _i)
    vp_a[idx] = vp_val
    vs_a[idx] = vs_val
    ro_a[idx] = ro_val

for k, v in zip(["VP", "VS", "RHO"], [vp_a, vs_a, ro_a]):
    mesh.attach_field(k, v)

mesh_3layer = sn.toolbox.detect_fluid(mesh)
print("Three-layer mesh built.")
print(f"  SLab: y = [{slab_top:.2f}, {slab_bottom:.2f}] m, vs={vs[0]:.0f} m/s")
print(f"  Weak layer: y = [{wl_top:.2f}, {wl_bottom:.2f}] m, vs={vs[1]:.0f} m/s")
print(f"  Air layer:  y = [{air_top:.2f}, {air_bottom:.2f}] m, vs={vs[2]:.0f} m/s")


In [ ]:
# # checking what is available in srcs dict
# print(srcs[0].get_dictionary()['source_time_function'].keys())
# print(srcs[0].get_dictionary()['source_time_function']['wavelet'][:10])  # print first 10 values of the source time function array
# # Print all available attributes and methods on the STF object
# print(dir(srcs[0].source_time_function))
# # Print only user-accessible attributes of the STF
# for attribute in dir(srcs[0].source_time_function):
#     if not attribute.startswith('_'):
#         print(attribute)


In [ ]:
# Initialize simulation object, pass multiple sources
sim = sc.simulation.Waveform(mesh=mesh_3layer, sources=srcs)

# physics parameters 
sim.physics.wave_equation.end_time_in_seconds = 2.5
sim.physics.wave_equation.start_time_in_seconds = 0.05

sim.output.volume_data.format = "hdf5"
sim.output.volume_data.fields = ["displacement", "velocity"]
sim.output.volume_data.filename = "volume_data_output.h5"
sim.output.volume_data.sampling_interval_in_time_steps = 50


moving_source_output_folder = str(pathlib.Path(PROJECT_DIR) / "custom_job_moving_source_all_sources")

print(f"Launching simulation. Outputs will be copied to: {moving_source_output_folder}")

# Launchg
salvus.flow.api.run(
    input_file=sim,
    site_name=SALVUS_FLOW_SITE_NAME,
    ranks=RANKS_PER_JOB,
    output_folder=moving_source_output_folder,
    overwrite=True,
    get_all=True, 
)

print("Run finished successfully!")

In [ ]:
# Extract velocity wavefield output from the single combined run.
vol_file = pathlib.Path("/mnt/sda/salome/heavy-workspace/Test Simulations/simulation_wavefield_custom_SUPER_stf_rolled/custom_job_moving_source_all_sources/volume_data_output.h5")
if not vol_file.exists():
    raise RuntimeError(
        f"Missing output file: {vol_file}. Re-run the launch cell and verify the job completed."
    )

vel_wo = wavefield_output.WavefieldOutput.from_file(
    vol_file,
    "velocity",
    "volume",
)

vel_2d_layered = wavefield_output.wavefield_output_to_xarray(
    vel_wo,
    points=[np.linspace(0, 400, 1601), np.linspace(0, 3, 101)],
)

print(f"Loaded velocity from {vol_file}")
print(f"Shape: {vel_2d_layered.dims}")
print(vel_2d_layered)

In [ ]:
# Force inline display in case backend was set to Agg earlier
import matplotlib.pyplot as plt
from IPython.display import display

# Define receiver line 
y_surface = 2.625 # 

# dis_2d_layered is already an xarray built from all events.
# Use direct selection instead of calling wavefield_output_to_xarray again.
    
# Resolve coordinate / dimension names robustly.
coords_set = set(vel_2d_layered.coords)
dims_set = set(vel_2d_layered.dims)

x_name = next((n for n in ["x", "X", "p0", "dim_0"] if n in coords_set or n in dims_set), None)
y_name = next((n for n in ["y", "Y", "p1", "dim_1"] if n in coords_set or n in dims_set), None)
t_name = next((n for n in ["t", "time"] if n in coords_set or n in dims_set), None)
c_name = next((n for n in ["c", "component"] if n in coords_set or n in dims_set), None)
e_name = next((n for n in ["event_index", "event", "source"] if n in coords_set or n in dims_set), None)

if x_name is None or y_name is None or t_name is None or c_name is None:
    raise ValueError(f"Could not resolve required dimensions. Found dims={vel_2d_layered.dims}, coords={list(vel_2d_layered.coords)}")

# Select both components at y receiver line
sg_vx = vel_2d_layered.isel({c_name: 0}).sel({y_name: y_surface}, method="nearest")
sg_vy = vel_2d_layered.isel({c_name: 1}).sel({y_name: y_surface}, method="nearest")

# Keep all moving-source events by reducing event dimension to a single stacked section.
if e_name is not None and e_name in sg_vx.dims:
    sg_vx_plot = sg_vx.mean(dim=e_name)
    sg_vy_plot = sg_vy.mean(dim=e_name)
else:
    sg_vx_plot = sg_vx
    sg_vy_plot = sg_vy

# Ensure plotting order is (time, x)
sg_vx_plot = sg_vx_plot.transpose(t_name, x_name)
sg_vy_plot = sg_vy_plot.transpose(t_name, x_name)

t_vals = sg_vx_plot[t_name].values
x_line = sg_vx_plot[x_name].values
data_vx = sg_vx_plot.values
data_vy = sg_vy_plot.values

print("Using dimensions:", {"x": x_name, "y": y_name, "t": t_name, "c": c_name, "event": e_name})
print("t range:", t_vals[0], "->", t_vals[-1])
print("vx shape:", data_vx.shape)
print("vy shape:", data_vy.shape)

# Clip colorscale (use same scale for both for comparison)
vmax = np.percentile(np.abs([data_vx, data_vy]), 95)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot vx (horizontal component)
im_vx = axes[0].pcolormesh(
    x_line,
    t_vals,
    data_vx,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[0].invert_yaxis()

# Mark all moving source positions
# for x_src in x_positions:
#     axes[0].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[0].set_xlabel("Distance (m)")
axes[0].set_ylabel("Time (s)")
axes[0].set_xlim(0, 400)
axes[0].set_title("Velocity(vx): horizontal component - receiver at crack interface")
plt.colorbar(im_vx, ax=axes[0], label="Velocity (vx)")

# Plot vy (vertical component)
im_vy = axes[1].pcolormesh(
    x_line,
    t_vals,
    data_vy,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[1].invert_yaxis()

# # Mark all moving source positions
# for x_src in x_positions:
#     axes[1].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[1].set_xlabel("Distance (m)")
axes[1].set_ylabel("Time (s)")
axes[1].set_xlim(0, 400)
axes[1].set_title("Velocity(vy): vertical component - receiver at crack interface")
plt.colorbar(im_vy, ax=axes[1], label="Velocity (vy)")

plt.tight_layout()

# Display explicitly in notebook output
display(fig)
plt.close(fig)

In [ ]:
# displacemnt and strain
moving_source_output_folder = str(pathlib.Path(PROJECT_DIR) / "custom_job_moving_source_all_sources")
# Extract displacement wavefield output from the single combined run.
vol_file = pathlib.Path(moving_source_output_folder) / "volume_data_output.h5"
if not vol_file.exists():
    raise RuntimeError(
        f"Missing output file: {vol_file}. Re-run the launch cell and verify the job completed."
    )

dis_wo = wavefield_output.WavefieldOutput.from_file(
    vol_file,
    "displacement",
    "volume",
)

dis_2d_layered = wavefield_output.wavefield_output_to_xarray(
    dis_wo,
    points=[np.linspace(0, 400, 1601), np.linspace(0, 3, 101)],
)

print(f"Loaded displacement from {vol_file}")
print(f"Shape: {dis_2d_layered.dims}")
print(dis_2d_layered)

print(f"WavefieldOutput shape: {dis_wo.data.shape}")
print(f"Shape: {dis_2d_layered.dims}")
print(dis_2d_layered)
print("WavefieldOutput shape:", dis_wo.data.shape)#
print("xarray shape:", dis_2d_layered.shape) 

# Take the spatial derivative of the layered displacement field along x.
x_dim = next((dim for dim in ["x", "X", "p0", "dim_0"] if dim in dis_2d_layered.dims or dim in dis_2d_layered.coords), None)

if x_dim is None:
    raise ValueError(
        f"Could not find an x-like spatial dimension in dims={dis_2d_layered.dims}, coords={list(dis_2d_layered.coords)}"
    )

dis_2d_layered_dx = dis_2d_layered.differentiate(x_dim)

print(f"Computed spatial derivative d/d{x_dim} for dis_2d_layered")
print(dis_2d_layered_dx)


In [ ]:
# plotting displacement
# Force inline display in case backend was set to Agg earlier
import matplotlib.pyplot as plt
from IPython.display import display

# Define receiver line 
y_surface = 2.625 

# dis_2d_layered is already an xarray built from all events.
# Use direct selection instead of calling wavefield_output_to_xarray again.
    
# Resolve coordinate / dimension names robustly.
coords_set = set(vel_2d_layered.coords)
dims_set = set(vel_2d_layered.dims)

x_name = next((n for n in ["x", "X", "p0", "dim_0"] if n in coords_set or n in dims_set), None)
y_name = next((n for n in ["y", "Y", "p1", "dim_1"] if n in coords_set or n in dims_set), None)
t_name = next((n for n in ["t", "time"] if n in coords_set or n in dims_set), None)
c_name = next((n for n in ["c", "component"] if n in coords_set or n in dims_set), None)
e_name = next((n for n in ["event_index", "event", "source"] if n in coords_set or n in dims_set), None)

if x_name is None or y_name is None or t_name is None or c_name is None:
    raise ValueError(f"Could not resolve required dimensions. Found dims={vel_2d_layered.dims}, coords={list(vel_2d_layered.coords)}")

# Select both components at y receiver line
dis_vx = dis_2d_layered.isel({c_name: 0}).sel({y_name: y_surface}, method="nearest")
dis_vy = dis_2d_layered.isel({c_name: 1}).sel({y_name: y_surface}, method="nearest")

# Keep all moving-source events by reducing event dimension to a single stacked section.
if e_name is not None and e_name in dis_vx.dims:
    dis_vx_plot = dis_vx.mean(dim=e_name)
    dis_vy_plot = dis_vy.mean(dim=e_name)
else:
    dis_vx_plot = dis_vx
    dis_vy_plot = dis_vy

# Ensure plotting order is (time, x)
dis_vx_plot = dis_vx_plot.transpose(t_name, x_name)
dis_vy_plot = dis_vy_plot.transpose(t_name, x_name)

t_vals = dis_vx_plot[t_name].values
x_line = dis_vx_plot[x_name].values
data_vx = dis_vx_plot.values
data_vy = dis_vy_plot.values

print("Using dimensions:", {"x": x_name, "y": y_name, "t": t_name, "c": c_name, "event": e_name})
print("t range:", t_vals[0], "->", t_vals[-1])
print("vx shape:", data_vx.shape)
print("vy shape:", data_vy.shape)

# Clip colorscale (use same scale for both for comparison)
vmax = np.percentile(np.abs([data_vx, data_vy]), 95)
#vmax = 1e-7

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot vx (horizontal component)
im_vx = axes[0].pcolormesh(
    x_line,
    t_vals,
    data_vx,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[0].invert_yaxis()

# Mark all moving source positions
# for x_src in x_positions:
#     axes[0].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[0].set_xlabel("Distance (m)")
axes[0].set_ylabel("Time (s)")
axes[0].set_xlim(0, 400)
axes[0].set_title("DIsplacement: horizontal component - receiver at crack interface")
plt.colorbar(im_vx, ax=axes[0], label="Displacement (vx)")

# Plot vy (vertical component)
im_vy = axes[1].pcolormesh(
    x_line,
    t_vals,
    data_vy,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[1].invert_yaxis()

# # Mark all moving source positions
# for x_src in x_positions:
#     axes[1].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[1].set_xlabel("Distance (m)")
axes[1].set_ylabel("Time (s)")
axes[1].set_xlim(0, 400)
axes[1].set_title("DIsplacement: vertical component - receiver at crack interface")
plt.colorbar(im_vy, ax=axes[1], label="Displacement (vy)")

plt.tight_layout()

# Display explicitly in notebook output
display(fig)
plt.close(fig)

In [ ]:
# instead of velocity: temporal derivative of displacement to get velocity
dis_2d_layered_dt = dis_2d_layered.differentiate(t_name)

# plotting displacement temporal derivative (velocity)
# Force inline display in case backend was set to Agg earlier
import matplotlib.pyplot as plt
from IPython.display import display

# Define receiver line 
y_surface = 2.625 # 

# dis_2d_layered is already an xarray built from all events.
# Use direct selection instead of calling wavefield_output_to_xarray again.
    
# Resolve coordinate / dimension names robustly.
coords_set = set(dis_2d_layered_dt.coords)
dims_set = set(dis_2d_layered_dt.dims)

x_name = next((n for n in ["x", "X", "p0", "dim_0"] if n in coords_set or n in dims_set), None)
y_name = next((n for n in ["y", "Y", "p1", "dim_1"] if n in coords_set or n in dims_set), None)
t_name = next((n for n in ["t", "time"] if n in coords_set or n in dims_set), None)
c_name = next((n for n in ["c", "component"] if n in coords_set or n in dims_set), None)
e_name = next((n for n in ["event_index", "event", "source"] if n in coords_set or n in dims_set), None)

if x_name is None or y_name is None or t_name is None or c_name is None:
    raise ValueError(f"Could not resolve required dimensions. Found dims={dis_2d_layered_dt.dims}, coords={list(dis_2d_layered_dt.coords)}")

# Select both components at y receiver line
sg_vx = dis_2d_layered_dt.isel({c_name: 0}).sel({y_name: y_surface}, method="nearest")
sg_vy = dis_2d_layered_dt.isel({c_name: 1}).sel({y_name: y_surface}, method="nearest")

# Keep all moving-source events by reducing event dimension to a single stacked section.
if e_name is not None and e_name in sg_vx.dims:
    sg_vx_plot = sg_vx.mean(dim=e_name)
    sg_vy_plot = sg_vy.mean(dim=e_name)
else:
    sg_vx_plot = sg_vx
    sg_vy_plot = sg_vy

# Ensure plotting order is (time, x)
sg_vx_plot = sg_vx_plot.transpose(t_name, x_name)
sg_vy_plot = sg_vy_plot.transpose(t_name, x_name)

t_vals = sg_vx_plot[t_name].values
x_line = sg_vx_plot[x_name].values
data_vx = sg_vx_plot.values
data_vy = sg_vy_plot.values

print("Using dimensions:", {"x": x_name, "y": y_name, "t": t_name, "c": c_name, "event": e_name})
print("t range:", t_vals[0], "->", t_vals[-1])
print("vx shape:", data_vx.shape)
print("vy shape:", data_vy.shape)

# Clip colorscale (use same scale for both for comparison)
vmax = np.percentile(np.abs([data_vx, data_vy]), 95)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot vx (horizontal component)
im_vx = axes[0].pcolormesh(
    x_line,
    t_vals,
    data_vx,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[0].invert_yaxis()

# Mark all moving source positions
# for x_src in x_positions:
#     axes[0].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[0].set_xlabel("Distance (m)")
axes[0].set_ylabel("Time (s)")
axes[0].set_xlim(0, 400)
axes[0].set_title("Velocity(vx) from displacement derivative: horizontal component - receiver at crack interface")
plt.colorbar(im_vx, ax=axes[0], label="Velocity (vx)")

# Plot vy (vertical component)
im_vy = axes[1].pcolormesh(
    x_line,
    t_vals,
    data_vy,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[1].invert_yaxis()

# # Mark all moving source positions
# for x_src in x_positions:
#     axes[1].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[1].set_xlabel("Distance (m)")
axes[1].set_ylabel("Time (s)")
axes[1].set_xlim(0, 400)
axes[1].set_title("Velocity(vy) from displacement derivative: vertical component - receiver at crack interface")
plt.colorbar(im_vy, ax=axes[1], label="Velocity (vy)")

plt.tight_layout()

# Display explicitly in notebook output
display(fig)
plt.close(fig)

In [ ]:
# Plottining strain for two different reciever lines 
# Force inline display in case backend was set to Agg earlier
import matplotlib.pyplot as plt
from IPython.display import display

# Define receiver line at snow-earth interface (model coordinates)
y_surface_1 = 2.25
y_surface_2 = 2.625

# dis_2d_layered is already an xarray built from all events.
# Use direct selection instead of calling wavefield_output_to_xarray again.
    
# Resolve coordinate / dimension names robustly.
coords_set = set(vel_2d_layered.coords)
dims_set = set(vel_2d_layered.dims)

x_name = next((n for n in ["x", "X", "p0", "dim_0"] if n in coords_set or n in dims_set), None)
y_name = next((n for n in ["y", "Y", "p1", "dim_1"] if n in coords_set or n in dims_set), None)
t_name = next((n for n in ["t", "time"] if n in coords_set or n in dims_set), None)
c_name = next((n for n in ["c", "component"] if n in coords_set or n in dims_set), None)
e_name = next((n for n in ["event_index", "event", "source"] if n in coords_set or n in dims_set), None)

if x_name is None or y_name is None or t_name is None or c_name is None:
    raise ValueError(f"Could not resolve required dimensions. Found dims={vel_2d_layered.dims}, coords={list(vel_2d_layered.coords)}")

# Select both components at y receiver line
sg_strain_y_1 = dis_2d_layered_dx.isel({c_name: 0}).sel({y_name: y_surface_1}, method="nearest")
sg_strain_y_2 = dis_2d_layered_dx.isel({c_name: 0}).sel({y_name: y_surface_2}, method="nearest")

# Keep all moving-source events by reducing event dimension to a single stacked section.
if e_name is not None and e_name in sg_strain_y_1.dims:
    sg_strain_y_1_plot = sg_strain_y_1.mean(dim=e_name)
    sg_strain_y_2_plot = sg_strain_y_2.mean(dim=e_name)
else:
    sg_strain_y_1_plot = sg_strain_y_1
    sg_strain_y_2_plot = sg_strain_y_2

# Ensure plotting order is (time, x)
sg_strain_y_1_plot = sg_strain_y_1_plot.transpose(t_name, x_name)
sg_strain_y_2_plot = sg_strain_y_2_plot.transpose(t_name, x_name)

t_vals = sg_strain_y_1_plot[t_name].values
x_line = sg_strain_y_1_plot[x_name].values
data_vx = sg_strain_y_1_plot.values
data_vy = sg_strain_y_2_plot.values

print("Using dimensions:", {"x": x_name, "y": y_name, "t": t_name, "c": c_name, "event": e_name})
print("t range:", t_vals[0], "->", t_vals[-1])
print("vx shape:", data_vx.shape)
print("vy shape:", data_vy.shape)

# Clip colorscale (use same scale for both for comparison)
vmax = np.percentile(np.abs([data_vx, data_vy]), 95)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot vx (horizontal component)
im_vx = axes[0].pcolormesh(
    x_line,
    t_vals,
    data_vx,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[0].invert_yaxis()

# Mark all moving source positions
# for x_src in x_positions:
#     axes[0].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[0].set_xlabel("Distance (m)")
axes[0].set_ylabel("Time (s)")
axes[0].set_xlim(0, 400)
axes[0].set_title("Strain (y): vertical component - receiver at snow bottom")
plt.colorbar(im_vx, ax=axes[0], label="Strain (y)")

# Plot vy (vertical component)
im_vy = axes[1].pcolormesh(
    x_line,
    t_vals,
    data_vy,
    shading="none",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[1].invert_yaxis()

# # Mark all moving source positions
# for x_src in x_positions:
#     axes[1].axvline(x=x_src, color="teal", lw=0.5, linestyle="--", alpha=0.35)

axes[1].set_xlabel("Distance (m)")
axes[1].set_ylabel("Time (s)")
axes[1].set_xlim(0, 400)
axes[1].set_title("Strain (y): vertical component - receiver at crack depth")
plt.colorbar(im_vy, ax=axes[1], label="Strain (y)")

plt.tight_layout()

# Display explicitly in notebook output
display(fig)
plt.close(fig)


In [ ]:
# Frequency domain analysis: f-k analysis with symmetric V-line overlays
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors
from IPython.display import display

# Verify rupture velocity
print(f"Rupture speed taken is {target_vprop} m/s")

# Surface shot gather selection
y_recv = 2.25   # Snow coordinate target position
# FIX: Changed .isel to .sel so method="nearest" handles physical coordinate mapping correctly
sg = vel_2d_layered.sel(
    {y_name: y_recv},
    method="nearest"
).isel({c_name: 0}) # vx component

t_vals = sg[t_name].values
x_vals = sg[x_name].values
data = sg.values 

# Force data strictly into (nt, nx) shape
if data.shape[0] == len(x_vals):
    data = data.T

nt, nx = data.shape
dt = float(np.nanmedian(np.diff(t_vals)))
dx = float(np.nanmedian(np.diff(x_vals)))

# 2D FFT - f-k spectrum with Hanning Windows
taper_t = np.hanning(nt)
taper_x = np.hanning(nx)
data_tapered = data * taper_t[:, None] * taper_x[None, :]

fk = np.fft.fftshift(np.fft.fft2(data_tapered))
freqs = np.fft.fftshift(np.fft.fftfreq(nt, dt))
wns = np.fft.fftshift(np.fft.fftfreq(nx, dx)) # wavenumber (cycles/m)

# Slice positive frequencies exclusively
f_pos_mask = freqs >= 0
fk_pos = fk[f_pos_mask, :]
# FIX: Properly slice frequency coordinate axis to match pk dimensions
f_plot = freqs[f_pos_mask]

# Define velocity regimes to track
velocities = {
    "vs_slab = 150 m/s": 150.0,
    "rupture speed": target_vprop,
    "vp_slab = 300 m/s": 300.0,
}
# Match colors array exactly to our 3 dictionary objects
colors = ["black", "blue", "green"]

fig, ax = plt.subplots(figsize=(12, 7))

# Build explicit 2D coordinate meshgrid for flawless pcolormesh mapping
WNS, FREQS = np.meshgrid(wns, f_plot)

# Compute power log spectrum
pk = np.log10(np.abs(fk_pos) + 1e-10)
vmax_pk = np.percentile(pk, 99)
vmin_pk = vmax_pk - 4 # 4 decades of dynamic range

mesh = ax.pcolormesh(
    WNS, FREQS, pk,
    shading="nearest", cmap="RdPu",
    vmin=vmin_pk, vmax=vmax_pk,
)

# Overlay symmetric velocity slopes on both sides of the center (k=0) vertex
k_max = wns.max()
for (label, v), col in zip(velocities.items(), colors):
    # Prograde Slope (Right side of V: k = +f/v)
    k_right = f_plot / v
    mask_right = np.abs(k_right) <= k_max
    ax.plot(k_right[mask_right], f_plot[mask_right], color=col, lw=1.8,
            linestyle="--", label=label)
    
    # Retrograde Slope (Left side of V: k = -f/v)
    # Uses same positive f_plot frequency domain to create full opposing slopes
    k_left = -f_plot / v
    mask_left = np.abs(k_left) <= k_max
    ax.plot(k_left[mask_left], f_plot[mask_left], color=col, lw=1.8,
            linestyle="--") # Omit label here so it doesn't duplicate in legend

ax.set_xlim(-k_max * 0.4, k_max * 0.4)
ax.set_ylim(0, min(3 * f0, freqs.max()))
ax.set_xlabel("Wavenumber k (cycles/m)")
ax.set_ylabel("Frequency f (Hz)")
ax.set_title("Symmetric f-k Spectrum of Surface vx — Rupture Velocity Identifications")
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.15, color="white", linestyle=":")

# Plot matching interactive colorbar mapping
cbar = fig.colorbar(mesh, ax=ax, shrink=0.9)
cbar.set_label("Relative Power Log10(Mag)", labelpad=10)

plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# =============================================================================
# UNIFIED WAVEFIELD ANIMATION ENGINE (Vy & Vx)
# =============================================================================
import pathlib
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import display
from salvus.toolbox.helpers import wavefield_output

# Always reload the latest simulation output from disk
base_dir = globals().get('PROJECT_DIR', "simulation_custom_stf_rolled")
project_path = pathlib.Path(base_dir) / "custom_job_moving_source_all_sources"

output_candidates = sorted(project_path.glob("**/volume_data_output.h5"), key=lambda path: path.stat().st_mtime, reverse=True)
if not output_candidates:
    raise RuntimeError(f"Missing output file under {project_path}: volume_data_output.h5")
vol_file = output_candidates[0]
print(f"Using latest wavefield file: {vol_file}")

vel_wo = wavefield_output.WavefieldOutput.from_file(vol_file, "velocity", "volume")
vel_2d_layered = wavefield_output.wavefield_output_to_xarray(
    vel_wo,
    points=[np.linspace(0, 400, 401), np.linspace(0, 3, 101)],
)

# Resolve coordinate and dimension naming conventions robustly
coords_set = set(vel_2d_layered.coords)
dims_set = set(vel_2d_layered.dims)
x_name = next((n for n in ["x", "X", "p0", "dim_0"] if n in coords_set or n in dims_set), None)
y_name = next((n for n in ["y", "Y", "p1", "dim_1"] if n in coords_set or n in dims_set), None)
t_name = next((n for n in ["t", "time"] if n in coords_set or n in dims_set), None)
c_name = next((n for n in ["c", "component"] if n in coords_set or n in dims_set), None)
e_name = next((n for n in ["event_index", "event", "source"] if n in coords_set or n in dims_set), None)

if None in [x_name, y_name, t_name, c_name]:
    raise ValueError(f"Could not resolve dimensions from dims={vel_2d_layered.dims}, coords={list(vel_2d_layered.coords)}")

receiver_y = 2.625

# =============================================================================
# PART 1: VERTICAL VELOCITY COMPONENT (vy)
# =============================================================================
print("\n--- Processing Animation 1/2: Velocity (vy) ---")
vy_event = vel_2d_layered.isel({c_name: 1})  # Component 1 = vy
if e_name is not None and e_name in vy_event.dims:
    vy_event = vy_event.mean(dim=e_name)
vy_event = vy_event.transpose(t_name, y_name, x_name)

x_vals_y = vy_event[x_name].values
y_vals_y = vy_event[y_name].values
t_vals_y = vy_event[t_name].values
frames_3d_y = np.asarray(vy_event.values, dtype=np.float64)
frames_3d_y = np.nan_to_num(frames_3d_y, nan=0.0, posinf=0.0, neginf=0.0)

t_start_idx = np.searchsorted(t_vals_y, 0.0)
N = 8
raw_idx_y = np.arange(t_start_idx, frames_3d_y.shape[0], N)
max_frames = 350
t_idx_y = raw_idx_y[:max_frames]

# Robust symmetric color scaling
sample_abs_y = np.abs(frames_3d_y[t_idx_y]).ravel()
sample_abs_y = sample_abs_y[np.isfinite(sample_abs_y) & (sample_abs_y > 1e-14)]
vmax_y = max(float(np.percentile(sample_abs_y, 99.0)), 1e-8) if sample_abs_y.size > 0 else 1e-8
norm_y = matplotlib.colors.TwoSlopeNorm(vcenter=0.0, vmin=-vmax_y, vmax=vmax_y)

fig_y, ax_y = plt.subplots(figsize=(12, 4), constrained_layout=True)
im_y = ax_y.imshow(
    frames_3d_y[t_idx_y[0]],
    extent=[x_vals_y.min(), x_vals_y.max(), y_vals_y.max(), y_vals_y.min()],
    aspect="auto",
    cmap="RdBu_r",
    norm=norm_y,
    origin="upper",
    interpolation="nearest",
)

ax_y.axhline(1.5, color="black", lw=1.2, linestyle="--", label=f"snow-air  (y=1.5 m)")
ax_y.set_xlabel("x (m)")
ax_y.set_ylabel("Depth (m)")
ax_y.set_xlim(x_vals_y.min(), x_vals_y.max())
ax_y.set_ylim(y_vals_y.max(), y_vals_y.min())
ax_y.legend(loc="upper right", fontsize=8)
plt.colorbar(im_y, ax=ax_y, label="Velocity (vy)", shrink=0.8)
title_y = ax_y.set_title(f"Wavefield Velocity(vy) - t = {t_vals_y[t_idx_y[0]]:.4f} s at snow bottom (y=2.25 m)")

def update_y(frame_idx):
    ti = t_idx_y[frame_idx]
    im_y.set_data(frames_3d_y[ti])
    title_y.set_text(f"Wavefield Velocity(vy) - t = {t_vals_y[ti]:.4f} s")
    return im_y, title_y

ani_y = animation.FuncAnimation(fig_y, update_y, frames=len(t_idx_y), interval=90, blit=False)
output_name_y = "custom_sub_wavefield_2d_moving_vy_crack.gif"
print(f"Saving {output_name_y}...")
ani_y.save(output_name_y, writer=animation.PillowWriter(fps=15), dpi=80)
plt.close(fig_y)
print(f"Done! Saved as {output_name_y}")


# =============================================================================
# PART 2: HORIZONTAL VELOCITY COMPONENT (vx)
# =============================================================================
print("\n--- Processing Animation 2/2: Velocity (vx) ---")
vx_event = vel_2d_layered.isel({c_name: 0})  # Component 0 = vx
if e_name is not None and e_name in vx_event.dims:
    vx_event = vx_event.mean(dim=e_name)
vx_event = vx_event.transpose(t_name, y_name, x_name)

# Arrays are extracted purely from the independent vx_event structure
x_vals_x = vx_event[x_name].values
y_vals_x = vx_event[y_name].values
t_vals_x = vx_event[t_name].values
frames_3d_x = np.asarray(vx_event.values, dtype=np.float64)
frames_3d_x = np.nan_to_num(frames_3d_x, nan=0.0, posinf=0.0, neginf=0.0)

raw_idx_x = np.arange(np.searchsorted(t_vals_x, 0.0), frames_3d_x.shape[0], N)
t_idx_x = raw_idx_x[:max_frames]

sample_abs_x = np.abs(frames_3d_x[t_idx_x]).ravel()
sample_abs_x = sample_abs_x[np.isfinite(sample_abs_x) & (sample_abs_x > 1e-14)]
vmax_x = max(float(np.percentile(sample_abs_x, 99.0)), 1e-8) if sample_abs_x.size > 0 else 1e-8
norm_x = matplotlib.colors.TwoSlopeNorm(vcenter=0.0, vmin=-vmax_x, vmax=vmax_x)

fig_x, ax_x = plt.subplots(figsize=(12, 4), constrained_layout=True)
im_x = ax_x.imshow(
    frames_3d_x[t_idx_x[0]],
    extent=[x_vals_x.min(), x_vals_x.max(), y_vals_x.max(), y_vals_x.min()],
    aspect="auto",
    cmap="RdBu_r",
    norm=norm_x,
    origin="upper",
    interpolation="nearest",
)

ax_x.axhline(1.5, color="black", lw=1.2, linestyle="--", label=f"snow-air  (y=1.5 m)")
ax_x.set_xlabel("x (m)")
ax_x.set_ylabel("Depth (m)")
ax_x.set_xlim(x_vals_x.min(), x_vals_x.max())
ax_x.set_ylim(y_vals_x.max(), y_vals_x.min())
ax_x.legend(loc="upper right", fontsize=8)
plt.colorbar(im_x, ax=ax_x, label="Velocity (vx)", shrink=0.8)
title_x = ax_x.set_title(f"Wavefield Velocity(vx) - t = {t_vals_x[t_idx_x[0]]:.4f} s at crack location (y = 2.625m)")

def update_x(frame_idx):
    ti = t_idx_x[frame_idx]
    im_x.set_data(frames_3d_x[ti])
    title_x.set_text(f"Wavefield Velocity(vx) - t = {t_vals_x[ti]:.4f} s")
    return im_x, title_x

ani_x = animation.FuncAnimation(fig_x, update_x, frames=len(t_idx_x), interval=90, blit=False)

if animation.writers.is_available("ffmpeg"):
    writer_x = animation.FFMpegWriter(
        fps=15,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p", "-crf", "28"],
    )
    output_name_x = "custom_sub_wavefield_2d_moving_vx_crack.mp4"
else:
    writer_x = animation.PillowWriter(fps=10)
    output_name_x = "custom_sub_wavefield_2d_moving_vx_crack.gif"

print(f"Saving {output_name_x}...")
ani_x.save(output_name_x, writer=writer_x, dpi=80)
plt.close(fig_x)
print(f"Done! Saved as {output_name_x}")